# International Football Results Analysis

This notebook answers the questions from Artificial Intelligence Exercise 1 using the international football results dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "results.csv"
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce")
df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce")
df.head()

## Basic Exploration

The full dataset is used for row counts, date range, and team counts.

In [ ]:
all_teams = pd.concat([df["home_team"], df["away_team"]], ignore_index=True)
basic_answers = pd.DataFrame({
    "question": [
        "Number of matches",
        "Earliest year",
        "Latest year",
        "Unique teams/countries",
        "Unique host countries",
        "Most frequent home team",
    ],
    "answer": [
        len(df),
        int(df["date"].dt.year.min()),
        int(df["date"].dt.year.max()),
        all_teams.nunique(),
        df["country"].nunique(),
        f"{df['home_team'].value_counts().idxmax()} ({df['home_team'].value_counts().max()} matches)",
    ]
})
basic_answers

## Goals Analysis

Future fixtures and rows without final scores are excluded from score analysis.

In [ ]:
completed = df.dropna(subset=["home_score", "away_score"]).copy()
completed["home_score"] = completed["home_score"].astype(int)
completed["away_score"] = completed["away_score"].astype(int)
completed["total_goals"] = completed["home_score"] + completed["away_score"]

highest = completed.sort_values(["total_goals", "date"], ascending=[False, True]).iloc[0]
goal_answers = pd.DataFrame({
    "question": [
        "Average goals per match",
        "Highest scoring match",
        "Home goals",
        "Away goals",
        "Most common total goals",
    ],
    "answer": [
        round(completed["total_goals"].mean(), 2),
        f"{highest['date'].date()}: {highest['home_team']} {highest['home_score']}-{highest['away_score']} {highest['away_team']}",
        int(completed["home_score"].sum()),
        int(completed["away_score"].sum()),
        int(completed["total_goals"].mode().iloc[0]),
    ]
})
goal_answers

## Match Results

In [ ]:
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    if row["home_score"] < row["away_score"]:
        return "Away Win"
    return "Draw"

completed["result"] = completed.apply(match_result, axis=1)
outcome_pct = (completed["result"].value_counts(normalize=True) * 100).round(2)
home_winners = completed.loc[completed["home_score"] > completed["away_score"], "home_team"]
away_winners = completed.loc[completed["away_score"] > completed["home_score"], "away_team"]
wins = pd.concat([home_winners, away_winners], ignore_index=True).value_counts()

result_answers = pd.DataFrame({
    "question": ["Home-win percentage", "Does home advantage exist?", "Team with most wins historically"],
    "answer": [
        f"{outcome_pct.get('Home Win', 0):.2f}%",
        "Yes, home teams win more often and score more total goals.",
        f"{wins.index[0]} ({wins.iloc[0]} wins)",
    ]
})
result_answers

## Visualizations

In [ ]:
completed["total_goals"].hist(bins=15)
plt.title("Distribution of Goals Per Match")
plt.xlabel("Total goals")
plt.ylabel("Matches")
plt.show()

In [ ]:
completed["result"].value_counts().plot(kind="bar")
plt.title("Match Outcomes")
plt.xlabel("Outcome")
plt.ylabel("Matches")
plt.show()

In [ ]:
wins.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Teams by Total Wins")
plt.xlabel("Wins")
plt.ylabel("Team")
plt.show()